# Simple trial-level artifact rejection

This notebook applies a deliberately simple and auditable rejection rule to the 19,040 trial-level tensors created in `all_trials_time_frequency.ipynb`.

For each exact eight-second trial, the corresponding 27-channel scalp EEG is filtered from 1–40 Hz and measured in µV. A trial is rejected when **any EEG electrode** has:

- peak-to-peak amplitude greater than **200 µV** (`extreme_amplitude`), or
- peak-to-peak amplitude less than **1 µV** (`flat_electrode`).

The rule does not use the left/right label, model prediction, participant performance, EOG, or EMG. Original tensors remain unchanged. Retained trials are copied into a separate artifact-rejected tensor collection, and every decision and measured value is recorded.

In [1]:
from pathlib import Path
import time

import mne
import numpy as np
import pandas as pd
from IPython.display import display

mne.set_log_level('ERROR')

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Signals.')

DATA_ROOT = find_project_data()
SIGNALS_ROOT = DATA_ROOT / 'processed' / 'Signals'
SOURCE_ROOT = DATA_ROOT / 'processed' / 'all_trials_time_frequency'
SOURCE_TENSOR_ROOT = SOURCE_ROOT / 'tensors_by_file'
SOURCE_TRIALS_PATH = SOURCE_ROOT / 'all_trials.csv'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'all_trials_time_frequency_artifact_rejected'
CLEAN_TENSOR_ROOT = OUTPUT_ROOT / 'tensors_by_file'
QUALITY_ROOT = OUTPUT_ROOT / 'quality_by_file'
MASTER_QUALITY_PATH = OUTPUT_ROOT / 'artifact_rejection_manifest.csv'
RETAINED_TRIALS_PATH = OUTPUT_ROOT / 'retained_trials.csv'
FILE_SUMMARY_PATH = OUTPUT_ROOT / 'file_summary.csv'

for directory in (CLEAN_TENSOR_ROOT, QUALITY_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

HIGH_PTP_THRESHOLD_UV = 200.0
FLAT_PTP_THRESHOLD_UV = 1.0
NON_EEG_CHANNELS = {'EOG1', 'EOG2', 'EOG3', 'EMGg', 'EMGd'}

all_trials = pd.read_csv(SOURCE_TRIALS_PATH)
source_files = sorted(all_trials['source_file'].unique())
assert len(all_trials) == 19_040
assert len(source_files) == 476

print(f'Source recordings: {len(source_files):,}')
print(f'Source trials: {len(all_trials):,}')
print(f'High-amplitude threshold: >{HIGH_PTP_THRESHOLD_UV:g} µV peak-to-peak')
print(f'Flat-electrode threshold: <{FLAT_PTP_THRESHOLD_UV:g} µV peak-to-peak')

Source recordings: 476
Source trials: 19,040
High-amplitude threshold: >200 µV peak-to-peak
Flat-electrode threshold: <1 µV peak-to-peak


In [2]:
def derivative_paths(source_file):
    relative = Path(source_file).with_suffix('')
    source_tensor = SOURCE_TENSOR_ROOT / relative.parent / f'{relative.name}_trial_ersp.npz'
    clean_tensor = CLEAN_TENSOR_ROOT / relative.parent / f'{relative.name}_trial_ersp_clean.npz'
    quality_csv = QUALITY_ROOT / relative.parent / f'{relative.name}_artifact_quality.csv'
    return source_tensor, clean_tensor, quality_csv

def atomic_csv(frame, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.tmp.csv')
    frame.to_csv(temporary, index=False)
    temporary.replace(destination)

def clean_outputs_are_valid(source_file):
    _, clean_tensor, quality_csv = derivative_paths(source_file)
    if not clean_tensor.is_file() or not quality_csv.is_file():
        return False
    try:
        quality = pd.read_csv(quality_csv)
        if len(quality) != 40:
            return False
        with np.load(clean_tensor, allow_pickle=False) as saved:
            retained = int(quality['retained'].sum())
            return (
                saved['X'].shape == (retained, 27, 23, 512)
                and len(saved['y']) == retained
                and float(saved['high_ptp_threshold_uv']) == HIGH_PTP_THRESHOLD_UV
                and float(saved['flat_ptp_threshold_uv']) == FLAT_PTP_THRESHOLD_UV
            )
    except Exception:
        return False

In [3]:
def reject_artifacts_for_recording(source_file, overwrite=False):
    source_tensor_path, clean_tensor_path, quality_path = derivative_paths(source_file)
    if not overwrite and clean_outputs_are_valid(source_file):
        quality = pd.read_csv(quality_path)
        return {
            'source_file': source_file, 'status': 'skipped_valid',
            'trials': len(quality), 'retained': int(quality['retained'].sum()),
            'rejected': int((~quality['retained']).sum()), 'error': '',
        }

    gdf_path = SIGNALS_ROOT / source_file
    trials = all_trials.loc[all_trials['source_file'].eq(source_file)].sort_values('trial').copy()
    if len(trials) != 40:
        raise ValueError(f'Expected 40 trial metadata rows, found {len(trials)}.')

    raw = mne.io.read_raw_gdf(gdf_path, preload=True, verbose='ERROR')
    eeg_channels = [channel for channel in raw.ch_names if channel not in NON_EEG_CHANNELS]
    if len(eeg_channels) != 27:
        raise ValueError(f'Expected 27 scalp EEG electrodes, found {len(eeg_channels)}.')
    raw.filter(1.0, 40.0, picks=eeg_channels, method='fir', phase='zero', verbose='ERROR')

    quality_records = []
    for row in trials.itertuples(index=False):
        eeg_uv = raw.get_data(
            picks=eeg_channels, start=int(row.start_sample), stop=int(row.end_sample)
        ) * 1e6
        channel_ptp_uv = np.ptp(eeg_uv, axis=1)
        maximum_index = int(np.argmax(channel_ptp_uv))
        minimum_index = int(np.argmin(channel_ptp_uv))
        maximum_ptp = float(channel_ptp_uv[maximum_index])
        minimum_ptp = float(channel_ptp_uv[minimum_index])
        extreme = maximum_ptp > HIGH_PTP_THRESHOLD_UV
        flat = minimum_ptp < FLAT_PTP_THRESHOLD_UV
        reasons = []
        if extreme:
            reasons.append('extreme_amplitude')
        if flat:
            reasons.append('flat_electrode')
        quality_records.append({
            'source_file': source_file,
            'dataset': row.dataset,
            'participant': row.participant,
            'run': row.run,
            'phase': row.phase,
            'trial': row.trial,
            'class_label': row.class_label,
            'max_ptp_uv': maximum_ptp,
            'max_ptp_channel': eeg_channels[maximum_index],
            'min_ptp_uv': minimum_ptp,
            'min_ptp_channel': eeg_channels[minimum_index],
            'extreme_amplitude': extreme,
            'flat_electrode': flat,
            'retained': not (extreme or flat),
            'rejection_reason': '|'.join(reasons),
            'high_ptp_threshold_uv': HIGH_PTP_THRESHOLD_UV,
            'flat_ptp_threshold_uv': FLAT_PTP_THRESHOLD_UV,
        })
    raw.close()

    quality = pd.DataFrame(quality_records)
    keep_mask = quality['retained'].to_numpy(dtype=bool)
    with np.load(source_tensor_path, allow_pickle=False) as source:
        if source['X'].shape[0] != 40 or not np.array_equal(source['trial_numbers'], trials['trial'].to_numpy()):
            raise ValueError('Tensor trial order does not match trial metadata.')
        payload = {key: source[key] for key in source.files}

    payload['X'] = payload['X'][keep_mask]
    payload['y'] = payload['y'][keep_mask]
    payload['trial_numbers'] = payload['trial_numbers'][keep_mask]
    payload['original_trial_indices'] = np.flatnonzero(keep_mask)
    payload['high_ptp_threshold_uv'] = np.asarray(HIGH_PTP_THRESHOLD_UV)
    payload['flat_ptp_threshold_uv'] = np.asarray(FLAT_PTP_THRESHOLD_UV)
    payload['artifact_method'] = np.asarray('1-40 Hz EEG peak-to-peak amplitude')

    clean_tensor_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_tensor = clean_tensor_path.with_name(clean_tensor_path.name + '.tmp.npz')
    np.savez_compressed(temporary_tensor, **payload)
    temporary_tensor.replace(clean_tensor_path)
    atomic_csv(quality, quality_path)

    retained = int(keep_mask.sum())
    return {
        'source_file': source_file, 'status': 'processed', 'trials': 40,
        'retained': retained, 'rejected': 40 - retained, 'error': '',
    }

## Smoke test

The clean A10 Run 3 demonstration is processed first. Its output is a valid checkpoint for the complete batch.

In [4]:
SMOKE_SOURCE = 'DATA A/A10/A10_R3_onlineT.gdf'
smoke_started = time.perf_counter()
smoke_result = reject_artifacts_for_recording(SMOKE_SOURCE, overwrite=False)
smoke_result['seconds'] = time.perf_counter() - smoke_started
display(pd.DataFrame([smoke_result]))
assert smoke_result['status'] in {'processed', 'skipped_valid'}

,source_file,status,trials,retained,rejected,error,seconds
0,DATA A/A10/A10_R3_onlineT.gdf,skipped_valid,40,40,0,,0.169198


## Apply rejection to every valid recording

The batch is resumable. Existing outputs are accepted only when the tensor, quality table, retained count, and both threshold values agree.

In [5]:
RUN_FULL_BATCH = True
OVERWRITE = False

batch_results = []
if RUN_FULL_BATCH:
    batch_started = time.perf_counter()
    for number, source_file in enumerate(source_files, start=1):
        started = time.perf_counter()
        try:
            result = reject_artifacts_for_recording(source_file, overwrite=OVERWRITE)
        except Exception as exc:
            result = {
                'source_file': source_file, 'status': 'failed', 'trials': 0,
                'retained': 0, 'rejected': 0,
                'error': f'{type(exc).__name__}: {exc}',
            }
        result['seconds'] = time.perf_counter() - started
        batch_results.append(result)

        if number == 1 or number % 10 == 0 or result['status'] == 'failed' or number == len(source_files):
            elapsed_minutes = (time.perf_counter() - batch_started) / 60
            print(
                f'[{number:>3}/{len(source_files)}] {result["status"]:<13} '
                f'retained={result["retained"]:>2} rejected={result["rejected"]:>2} '
                f'{source_file} ({elapsed_minutes:.1f} min)'
            )

    batch_log = pd.DataFrame(batch_results)
    batch_log.to_csv(OUTPUT_ROOT / 'processing_log.csv', index=False)
    display(batch_log['status'].value_counts().rename_axis('status').to_frame('recordings'))
    failures = batch_log.loc[batch_log['status'].eq('failed')]
    if not failures.empty:
        display(failures)
        raise RuntimeError(f'{len(failures)} recording(s) failed; valid checkpoints are safe.')
else:
    print('Full batch disabled; only the smoke-test output exists.')

[  1/476] skipped_valid retained=40 rejected= 0 DATA A/A1/A1_R2_acquisition.gdf (0.0 min)
[ 10/476] skipped_valid retained=40 rejected= 0 DATA A/A10/A10_R5_onlineT.gdf (0.0 min)
[ 20/476] skipped_valid retained=39 rejected= 1 DATA A/A12/A12_R3_onlineT.gdf (0.0 min)
[ 30/476] skipped_valid retained=35 rejected= 5 DATA A/A14/A14_R1_acquisition.gdf (0.1 min)
[ 40/476] skipped_valid retained=40 rejected= 0 DATA A/A15/A15_R5_onlineT.gdf (0.1 min)
[ 50/476] skipped_valid retained=40 rejected= 0 DATA A/A18/A18_R3_onlineT.gdf (0.1 min)
[ 60/476] skipped_valid retained=39 rejected= 1 DATA A/A2/A2_R1_acquisition.gdf (0.1 min)
[ 70/476] skipped_valid retained=31 rejected= 9 DATA A/A20/A20_R5_onlineT.gdf (0.2 min)
[ 80/476] skipped_valid retained=40 rejected= 0 DATA A/A22/A22_R3_onlineT.gdf (0.2 min)
[ 90/476] skipped_valid retained=39 rejected= 1 DATA A/A24/A24_R1_acquisition.gdf (0.2 min)
[100/476] skipped_valid retained=40 rejected= 0 DATA A/A25/A25_R5_onlineT.gdf (0.2 min)
[110/476] skipped_va

,recordings
status,
skipped_valid,476


## Consolidate and audit rejection decisions

In [6]:
invalid_outputs = [source_file for source_file in source_files if not clean_outputs_are_valid(source_file)]
if invalid_outputs:
    display(pd.DataFrame({'invalid_source_file': invalid_outputs}))
    raise RuntimeError(f'{len(invalid_outputs)} cleaned output(s) are missing or invalid.')

quality_frames = [pd.read_csv(derivative_paths(source_file)[2]) for source_file in source_files]
quality_manifest = pd.concat(quality_frames, ignore_index=True)
quality_manifest.to_csv(MASTER_QUALITY_PATH, index=False)

retained_keys = quality_manifest.loc[quality_manifest['retained'], ['source_file', 'trial']]
retained_trials = all_trials.merge(retained_keys, on=['source_file', 'trial'], how='inner', validate='one_to_one')
retained_trials.to_csv(RETAINED_TRIALS_PATH, index=False)

file_summary = (
    quality_manifest.groupby(['source_file', 'dataset', 'participant', 'run', 'phase'])
    .agg(
        trials=('trial', 'size'),
        retained=('retained', 'sum'),
        extreme_amplitude=('extreme_amplitude', 'sum'),
        flat_electrode=('flat_electrode', 'sum'),
        maximum_ptp_uv=('max_ptp_uv', 'max'),
    )
    .reset_index()
)
file_summary['rejected'] = file_summary['trials'] - file_summary['retained']
file_summary.to_csv(FILE_SUMMARY_PATH, index=False)

assert len(quality_manifest) == len(all_trials) == 19_040
assert len(retained_trials) == int(quality_manifest['retained'].sum())
assert set(quality_manifest['class_label']) == {'left', 'right'}

decision_summary = (
    quality_manifest.groupby(['phase', 'class_label'])['retained']
    .agg(['count', 'sum']).rename(columns={'count': 'trials', 'sum': 'retained'})
    .reset_index()
)
decision_summary['rejected'] = decision_summary['trials'] - decision_summary['retained']
decision_summary['rejection_rate'] = decision_summary['rejected'] / decision_summary['trials']
display(decision_summary)

print(f'Total trials:    {len(quality_manifest):,}')
print(f'Retained trials: {quality_manifest["retained"].sum():,}')
print(f'Rejected trials: {(~quality_manifest["retained"]).sum():,}')
print(f'Overall rejection rate: {(~quality_manifest["retained"]).mean():.2%}')
print(f'Extreme-amplitude trials: {quality_manifest["extreme_amplitude"].sum():,}')
print(f'Flat-electrode trials:    {quality_manifest["flat_electrode"].sum():,}')
print(f'Quality manifest: {MASTER_QUALITY_PATH}')
print(f'Clean tensors:    {CLEAN_TENSOR_ROOT}')

,phase,class_label,trials,retained,rejected,rejection_rate
0,acquisition,left,3180,2963,217,0.068239
1,acquisition,right,3180,2956,224,0.070440
2,online,left,6340,5844,496,0.078233
3,online,right,6340,5835,505,0.079653


Total trials:    19,040
Retained trials: 17,598
Rejected trials: 1,442
Overall rejection rate: 7.57%
Extreme-amplitude trials: 1,442
Flat-electrode trials:    0
Quality manifest: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/all_trials_time_frequency_artifact_rejected/artifact_rejection_manifest.csv
Clean tensors:    /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/all_trials_time_frequency_artifact_rejected/tensors_by_file


## Scope and limitation

Peak-to-peak rejection catches large transients and flat electrodes, but it does not reliably identify subtle eye movements, muscle activity, line noise, or unusual spatial patterns. It is appropriate as the simplest first-pass method, not as proof that every retained trial is artifact-free. Class balance and retained-trial counts must be handled explicitly during grouped model evaluation.